# 1. Fine-tune the furniture detector (YOLOv8 on HomeObjects-3K)
Open in Google Colab -> Runtime -> Change runtime type -> **T4 GPU**, then Run all (about 30-60 minutes).
The last cell downloads `furniture.pt`. Put it in `backend/perception/weights/furniture.pt` and restart the backend; the app then uses it automatically.

Note: HomeObjects-3K has beds, sofas, chairs, tables, lamps, etc. but **no separate "mattress" class**. It is trained on real beds, so it should confuse floors and folded mattresses less, but check it on your own photos.

In [ ]:
!pip -q install ultralytics

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")                       # start from COCO-pretrained weights
model.train(data="HomeObjects-3K.yaml",          # downloads the dataset (~390 MB) automatically
            epochs=60, imgsz=640, batch=16, patience=15, name="homeobjects")

In [ ]:
metrics = model.val()
print("mAP50:", round(metrics.box.map50, 3), " mAP50-95:", round(metrics.box.map, 3))
print("Classes the model knows:", model.names)   # copy the numbers + names into your report

In [ ]:
# Compare against the original COCO model on the same validation set (a good table for your report)
base = YOLO("yolov8s.pt").val(data="HomeObjects-3K.yaml")
print("COCO baseline mAP50:", round(base.box.map50, 3), "  fine-tuned mAP50:", round(metrics.box.map50, 3))

In [ ]:
from google.colab import files
files.download("runs/detect/homeobjects/weights/best.pt")   # rename it to furniture.pt on your PC